# Loan Credit-Risk Prediction

Predicts good vs bad credit risk from the classic German Credit dataset (UCI, 1,000 applicants, 20 features) — the core of loan-approval scoring.


In [1]:

import io, zipfile, urllib.request
import pandas as pd
url = "https://archive.ics.uci.edu/static/public/144/statlog+german+credit+data.zip"
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
zf = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(req).read()))
name = [n for n in zf.namelist() if n.endswith("german.data")][0]
cols = ["status","duration","credit_history","purpose","amount","savings","employment","installment_rate","personal_status","other_debtors","residence","property","age","other_installments","housing","existing_credits","job","dependents","telephone","foreign","risk"]
d = pd.read_csv(io.StringIO(zf.read(name).decode()), sep=r"\s+", header=None, names=cols)
print(d.shape); d.head()


(1000, 21)


,status,duration,credit_history,purpose,amount,savings,employment,installment_rate,personal_status,other_debtors,...,property,age,other_installments,housing,existing_credits,job,dependents,telephone,foreign,risk
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


In [2]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
y = (d["risk"] == 1).astype(int)  # 1 = good credit (approve), 2 = bad credit
X = pd.get_dummies(d.drop(columns=["risk"]), drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
models = {"Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
          "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42)}
for name, m in models.items():
    m.fit(X_train, y_train); print(name, "accuracy:", round(m.score(X_test, y_test), 4))
best_name = max(models, key=lambda n: models[n].score(X_test, y_test))
best_model = models[best_name]
print("Best:", best_name)


Logistic Regression accuracy: 0.724


Random Forest accuracy: 0.728
Best: Random Forest


In [3]:

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pred = best_model.predict(X_test)
acc = accuracy_score(y_test, pred)
print(f"{best_name} accuracy: {acc:.4f}")
print(classification_report(y_test, pred))
cm = confusion_matrix(y_test, pred)
fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Confusion matrix")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="white" if cm[i,j] > cm.max()/2 else "black")
plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=120); plt.show()


Random Forest accuracy: 0.7280
              precision    recall  f1-score   support

           0       0.59      0.29      0.39        75
           1       0.75      0.91      0.82       175

    accuracy                           0.73       250
   macro avg       0.67      0.60      0.61       250
weighted avg       0.70      0.73      0.70       250



## Result

See README.md for the verified accuracy. A teaching model on a classic dataset — real lending decisions need far more data, fairness testing and human review.
